# Create NER evaluation dataset

In [ ]:
%load_ext autoreload
%autoreload 2

## Init

In [ ]:
import logging
import sys
from logging.handlers import RotatingFileHandler
from pathlib import Path
from typing import Optional
import ast
import itertools
# Third-party imports (grouped for clarity)
# Note: Ensure all these are actually used in the script to avoid overhead
import base64
import difflib
import io
import json
import pprint
import random
import traceback
import urllib.request
from collections import Counter
from datetime import datetime

import argilla as rg
import html2text

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import regex as re
import requests
import seaborn as sns
from bs4 import BeautifulSoup
from dotenv import dotenv_values, find_dotenv
from huggingface_hub import DatasetCard, DatasetCardData
from tabulate import tabulate
from tqdm import tqdm

import spacy
from pathlib import Path
from wtpsplit import WtP, SaT

from typing import List, Union
from dataclasses import dataclass

# HuggingFace Datasets
from datasets import (Dataset, DatasetDict, DatasetInfo, Features, Sequence, Value, load_from_disk)

# Initialize Logger
import logging
from logging.config import dictConfig 
logger = logging.getLogger(__name__)
from archaeo_ner_greek.logging_config import LOGGING_CONFIG
dictConfig(LOGGING_CONFIG) 

# --- Configuration ---
PROJECT_NAME = "atrium-csd-ner"
# Uses the user's home directory dynamically
BASE_DIR = Path.home() / "src" /  PROJECT_NAME
DATA_DIR = BASE_DIR / "data"
LOG_DIR = Path.home() / "logs"

# Plotting setup
sns.set_theme()

# import itables
# itables.init_notebook_mode(all_interactive=False)

# --- Initialization ---
# Load environment variables
env_path = find_dotenv()
env_vars = dotenv_values()

# Create directories
DATA_DIR.mkdir(parents=True, exist_ok=True)

from archaeo_ner_greek.utils import configure_argilla_client, configure_argilla_resources, get_dataset_as_dataframe

logger.info(f"Finished initializing environment.")
logger.debug(f"Using {env_path}.")
logger.debug(f"Data directory set to: {DATA_DIR}")
logger.info(f"Environment variables: {env_vars}")


In [ ]:
from gliner2 import GLiNER2

# Load model once, use everywhere
# extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
extractor = GLiNER2.from_pretrained("fastino/gliner2-large-v1")


In [ ]:
def get_annotation_guidelines() -> dict:
    """
    Returns a dictionary of NER guidelines optimized for GLiNER 2.0.
    """
    return {
        "ARTEFACT": (
            "Artefacts: Portable objects made or modified by humans. "
            "Includes: 'pottery', 'shards', 'flint', 'charcoal', 'bones', 'coins'. "
            "Includes construction debris like 'CBM', 'tiles', 'bricks', and 'daub'. "
            "Rules: "
            "1. Do NOT annotate natural substances like 'water', 'sand', 'clay', 'soil', or 'yellow clay' as artefacts. "
            "2. Do NOT annotate features like 'holes' or 'pits' (these are Contexts). "
            "3. Include the material adjective if it describes the object (e.g., 'iron nail')."
        ),
        "PERIOD": (
            "Time Periods: Chronological eras, dates, and centuries. "
            "Examples: 'Late Neolithic', 'Modern', 'Roman', '2012', '19th century'. "
            "Do NOT annotate context codes like 'C404' or 'S22' or 'c561' or measurements like '20cm'."
        ),
        "LOCATION": (
            "Locations: Geographic names, cardinal directions used as locations, and addresses. "
            "Examples: 'North', 'South', 'Northeast corner', 'Zutphen'. "
            "Includes specific excavation areas like 'north side' or 'trench 1'."
        ),
        "CONTEXT": (
            "Contexts: Immobile features, structures, or excavation units. "
            "Includes: 'ditch', 'pit', 'grave', 'posthole', 'animal hole', 'mouse hole', 'wall'. "
            "Rules: Annotate the type of feature (e.g., 'hole'), not the ID number."
            "Do not annotate context codes like 'C404' or the word 'context'."
        ),
        "CONTEXT_ID": (
            "Context Identifiers: Alphanumeric codes for recording units. "
            "Examples: 'C404', 'S22', 'F05'. "
            "Rules: Must be a code, not a measurement (like '20cm') or a count (like '6~7')."
        ),
        "MATERIAL": (
            "Materials: The substance an artefact is made of. "
            "Examples: 'ceramic', 'iron', 'bronze', 'glass'. "
            "Rules: "
            "1. Only annotate if it refers to an object (e.g., 'gold' in 'gold coin'). "
            "2. Do NOT annotate environmental materials like 'water', 'soil', 'clay', 'mud', or 'sand' unless they are the raw material of a specific find."
        ),
        "SPECIES": (
            "Species: Biological names of animals, plants, and humans. "
            "Examples: 'cattle', 'pig', 'human', 'oak'. "
            "Includes generic terms like 'animal' (in 'animal bone')."
        ),
        "PERSON": (
            "Deities/Mythology: Names of gods or mythological figures only. "
            "Do NOT annotate researchers or modern people."
        ),
    }

### Configure the evaluation dataset



### Setup the dataset


In [ ]:

def setup_argilla_dataset(client: rg.Argilla, dataset_name: str, workspace_name: str, guidelines: str):
    """
    Recreates the Argilla dataset schema.
    - Removes 'document_sentence_id_field'
    - Adds 'context_sheet_description_id' as metadata
    """
    try:
        existing_dataset = client.datasets(name=dataset_name, workspace=workspace_name)
        if existing_dataset:
            logger.info(f"Deleting existing dataset: {dataset_name}")
            existing_dataset.delete()
    except Exception:
        pass
        
    try:
        settings = rg.Settings(
            guidelines=guidelines,
            fields=[
                rg.TextField(name="sentence_field", title="Context Description", required=True),
            ],
            metadata=[
                rg.IntegerMetadataProperty(name="context_sheet_description_id", title="Context Sheet Description ID", visible_for_annotators=True),
            ],
            questions=[
                rg.SpanQuestion(
                    name="entities",
                    title="Entities",
                    field="sentence_field",
                    labels=[
                        'ARTEFACT', 'PERIOD', "LOCATION", 'CONTEXT', 'CONTEXT_ID', 
                        'MATERIAL', 'SPECIES', "FEATURE", "PERSON", "MISC"
                    ],
                    allow_overlapping=True,
                ),
                rg.TextQuestion(
                    name="label_suggestion",
                    title="Label suggestion",
                    description="Suggest a new label for MISC items.",
                    required=False
                ),
            ],
        )
        
        dataset = rg.Dataset(
            name=dataset_name,
            workspace=workspace_name, 
            settings=settings,
            client=client,
        )
        dataset.create()
        logger.info(f"Dataset {dataset_name} created successfully.")
        return dataset

    except Exception:
        logger.error("Failed to setup Argilla dataset", exc_info=True)
        return None



#### Sample diverse lengths


In [ ]:
def sample_diverse_lengths(df: pd.DataFrame, n: int = 100, long_ratio: float = 0.8) -> pd.DataFrame:
    """
    Creates a sample of size n, comprised mostly of the longest texts 
    plus a random selection of shorter texts.

    Args:
        df: Source DataFrame.
        n: Total number of records to return.
        long_ratio: Percentage of the sample that should be 'long' texts (0.0 to 1.0).
                    Default 0.8 means 80 long texts and 20 short texts.
    """
    if len(df) <= n:
        logger.warning(f"Dataset size ({len(df)}) is smaller than requested sample ({n}). Returning full dataset.")
        return df

    # 1. Helper to calculate text length safely
    def get_text_len(row):
        fields = row.get("fields", {})
        if isinstance(fields, dict):
            text = fields.get("context_sheet_description_field", "")
            return len(str(text)) if text else 0
        return 0

    # 2. Create a temporary length column for sorting
    # We work on a copy to avoid SettingWithCopy warnings on the original df
    df_sample = df.copy()
    df_sample["_text_length"] = df_sample.apply(get_text_len, axis=1)

    # 3. Sort by length descending (Longest first)
    df_sorted = df_sample.sort_values(by="_text_length", ascending=False)

    # 4. Determine split indices
    # e.g., for n=100, ratio=0.8 -> we want 80 long, 20 short
    n_long = int(n * long_ratio)
    n_short = n - n_long

    # 5. Select the "Long" portion (Top N_long)
    df_long = df_sorted.iloc[:n_long]

    # 6. Select the "Short" portion (Random sample from the remaining rows)
    df_remaining = df_sorted.iloc[n_long:]
    
    # Handle edge case: if not enough remaining rows for the short sample
    if len(df_remaining) < n_short:
        df_short = df_remaining
    else:
        df_short = df_remaining.sample(n=n_short, random_state=42)

    # 7. Combine and Shuffle
    # We shuffle so the annotator doesn't see texts strictly ordered by length
    final_sample = pd.concat([df_long, df_short]).sample(frac=1, random_state=42)

    # Clean up temp column
    final_sample = final_sample.drop(columns=["_text_length"])

    logger.info(
        f"Created sample of {len(final_sample)} records: "
        f"{len(df_long)} longest texts + {len(df_short)} random shorter texts."
    )
    
    return final_sample

### Annotate records

In [ ]:
def debug_dataframe_structure(df):
    """
    Inspects the first row of the DataFrame to verify field structure and keys.
    """
    if df is None or len(df) == 0:
        logger.error("CRITICAL: DataFrame is empty or None.")
        return

    logger.info(f"DataFrame Shape: {df.shape}")
    
    if 'fields' not in df.columns:
        logger.error("CRITICAL: Column 'fields' not found in DataFrame.")
        return

    # Inspect the first row
    first_row_fields = df.iloc[0]['fields']
    logger.info(f"Type of 'fields' data: {type(first_row_fields)}")
    logger.info(f"Raw content of 'fields' (first row): {first_row_fields}")

    if isinstance(first_row_fields, dict):
        keys = list(first_row_fields.keys())
        logger.info(f"Available keys in 'fields': {keys}")
        
        target_key = "context_sheet_description_field"
        if target_key in keys:
            val = first_row_fields[target_key]
            logger.info(f"SUCCESS: Found key '{target_key}'. Content length: {len(str(val))}")
        else:
            logger.error(f"FAILURE: Key '{target_key}' NOT found. Please update the extraction code to use one of the available keys.")
    else:
        logger.warning("WARNING: 'fields' is not a dictionary. It might be a JSON string or other type.")

# Run this in a cell before annotation
# debug_dataframe_structure(df)


import logging
import pandas as pd
import numpy as np
from pathlib import Path

logger = logging.getLogger(__name__)

def patch_predictions_with_corrections(
    df: pd.DataFrame, 
    corrections_csv_path: Path
) -> pd.DataFrame:
    """
    Applies manual corrections from a CSV to the 'gliner_predictions' column.
    
    Logic:
    - Matches entities based on (id, start, end).
    - If new_label is 'None' (string): Deletes the entity.
    - If new_label is a valid string: Renames the entity.
    - If new_label is empty/NaN: Ignores the line (does nothing).
    """
    if not corrections_csv_path.exists():
        logger.warning(f"Corrections file not found at {corrections_csv_path}. Skipping patching.")
        return df

    # 1. Load Corrections
    # We define column names manually based on your provided structure
    col_names = [
        "doc_id", "text", "old_label", "score", 
        "start", "end", "context", "overlap", "new_label"
    ]
    
    try:
        corrections_df = pd.read_csv(corrections_csv_path, header=None, names=col_names)
    except Exception as e:
        logger.error(f"Failed to read corrections CSV: {e}")
        return df

    logger.info(f"Loaded {len(corrections_df)} correction rules.")

    # 2. Build Lookup Map
    # Key: (doc_id, start, end) -> Value: new_label
    # This ensures O(1) access time inside the nested loop
    corrections_map = {}
    
    for _, row in corrections_df.iterrows():
        # Skip if new_label is NaN (empty in CSV) -> "Do nothing"
        if pd.isna(row['new_label']):
            continue
            
        # Create a unique key for the entity
        key = (str(row['doc_id']).strip(), int(row['start']), int(row['end']))
        
        # specific string "None" handling
        val = str(row['new_label']).strip()
        corrections_map[key] = val

    # 3. Apply Patches
    patched_count = 0
    deleted_count = 0
    
    # We work on a copy to avoid SettingWithCopy warnings
    output_df = df.copy()

    for idx, row in output_df.iterrows():
        original_preds = row.get("gliner_predictions", [])
        if not original_preds:
            continue
            
        doc_id = str(row.get("id")).strip()
        new_preds = []
        
        for entity in original_preds:
            # Construct key to check against corrections
            ent_key = (doc_id, int(entity['start']), int(entity['end']))
            
            if ent_key in corrections_map:
                new_label = corrections_map[ent_key]
                
                if new_label == "None":
                    # Case: Remove entity
                    deleted_count += 1
                    continue # Skip appending to new_preds
                else:
                    # Case: Rename entity
                    entity['label'] = new_label
                    new_preds.append(entity)
                    patched_count += 1
            else:
                # No correction found, keep original
                new_preds.append(entity)
        
        # Update the row
        output_df.at[idx, "gliner_predictions"] = new_preds

    logger.info(f"Patching complete: {patched_count} entities renamed, {deleted_count} removed.")
    return output_df

def standardize_and_filter_predictions(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Parses GLiNER dictionary output.
    2. Converts text predictions to character offsets (start/end).
    3. Removes nested entities of the same type.
    """
    df_clean = df.copy()
    
    # Helper to extract text safely
    def get_text(row):
        fields = row.get("fields", {})
        if isinstance(fields, dict):
            return fields.get("context_sheet_description_field", "")
        return ""

    cleaned_preds_column = []

    logger.info("Converting and filtering predictions...")
    
    for idx, row in tqdm(df_clean.iterrows(), total=len(df_clean)):
        text = get_text(row)
        raw_preds = row.get("gliner_predictions", {})
        
        # --- A. Parse Input Format ---
        preds_dict = {}
        if isinstance(raw_preds, dict):
            preds_dict = raw_preds
        elif isinstance(raw_preds, str):
            try:
                # Clean and parse stringified dicts
                clean_str = raw_preds.replace("array(", "").replace(")", "")
                preds_dict = ast.literal_eval(clean_str)
            except (ValueError, SyntaxError):
                preds_dict = {}

        # --- B. Convert to Flat List of Spans ---
        flat_spans = []
        
        # Check if it's the Dictionary Format {'entities': {'LABEL': ['text']}}
        entities_group = preds_dict.get("entities", {})
        if isinstance(entities_group, dict):
            for label, entity_texts in entities_group.items():
                if not isinstance(entity_texts, list): continue
                for entity_text in entity_texts:
                    if not isinstance(entity_text, str) or not entity_text: continue
                    
                    # Find ALL occurrences of the entity text
                    pattern = re.escape(entity_text)
                    for match in re.finditer(pattern, text):
                        flat_spans.append({
                            "label": str(label),
                            "start": match.start(),
                            "end": match.end(),
                            "text": entity_text
                        })
        
        # --- C. Filter Nested Entities ---
        # Sort by Start (asc) then Length (desc) -> Longest parent comes first
        flat_spans.sort(key=lambda x: (x['start'], -1 * (x['end'] - x['start'])))
        
        kept_spans = []
        for candidate in flat_spans:
            is_nested = False
            for existing in kept_spans:
                # Check bounds: Candidate is inside Existing
                if (candidate['start'] >= existing['start'] and 
                    candidate['end'] <= existing['end']):
                    
                    # Check Label: Only remove if SAME label (e.g. PERIOD inside PERIOD)
                    if candidate['label'] == existing['label']:
                        is_nested = True
                        break
            
            if not is_nested:
                kept_spans.append(candidate)

        cleaned_preds_column.append(kept_spans)

    df_clean["gliner_predictions"] = cleaned_preds_column
    return df_clean

def annotate_dataframe(
    df: pd.DataFrame,
    extractor,
    guidelines_dict: dict[str, str],
    threshold: float = 0.3,
    batch_size: int = 16
) -> pd.DataFrame:
    """
    Annotates DataFrame with verbose logging for debugging.
    """
    output_df = df.copy()
    
    # Ensure the output column exists
    output_df["gliner_predictions"] = np.empty((len(output_df), 0)).tolist()

    valid_batch_data = []
    
    logger.info("--- STARTING DATA PREPROCESSING ---")
    
    # 1. Extraction Loop with Counters
    skipped_count = 0
    for idx, row in output_df.iterrows():
        fields_data = row.get("fields", {})
        text = None
        
        # Robust extraction (handles strings if JSON parsing is needed, though Argilla usually returns dicts)
        if isinstance(fields_data, dict):
            # --- CRITICAL: Ensure this key matches your debug output ---
            text = fields_data.get("context_sheet_description_field")
        
        if text and isinstance(text, str) and len(text.strip()) > 0:
            valid_batch_data.append((idx, text))
        else:
            skipped_count += 1
            # Log the first failure to help debug
            if skipped_count == 1:
                logger.debug(f"First skipped row info: Fields type={type(fields_data)}, Text extracted={text}")

    logger.info(f"Preprocessing Complete.")
    logger.info(f"Total rows: {len(output_df)}")
    logger.info(f"Valid texts queued: {len(valid_batch_data)}")
    logger.info(f"Skipped rows (empty/invalid): {skipped_count}")

    if not valid_batch_data:
        logger.error("STOPPING: No valid text fields found. Check your field keys!")
        return output_df

    # 2. Inference Loop
    logger.info(f"Starting Inference on {len(valid_batch_data)} items with batch size {batch_size}...")
    
    # Force tqdm to stdout to ensure visibility
    pbar = tqdm(total=len(valid_batch_data), desc="GLiNER Inference", unit="doc")
    
    for i in range(0, len(valid_batch_data), batch_size):
        batch = valid_batch_data[i : i + batch_size]
        batch_indices = [item[0] for item in batch]
        batch_texts = [item[1] for item in batch]
        
        batch_predictions = []

        # Log start of first batch to confirm model execution
        if i == 0:
            logger.info(f"Processing first batch of {len(batch_texts)} texts...")

        # Iterative inference (Safest for GLiNER)
        for text in batch_texts:
            try:
                entities = extractor.extract_entities(
                    text, 
                    guidelines_dict, 
                    threshold=threshold
                )
                logger.info(f"Entities extracted: {entities} from {text}")
                batch_predictions.append(entities)
            except Exception as e:
                logger.error(f"Inference failed for text: {str(e)[:100]}...")
                batch_predictions.append([])
        
        # Map back to DataFrame
        output_df.loc[batch_indices, "gliner_predictions"] = batch_predictions
        pbar.update(len(batch))

    pbar.close()


    logger.info("Annotation process completed successfully.")
    return output_df


### Log records to Argilla


In [ ]:


def log_annotation_records(df: pd.DataFrame, dataset: rg.Dataset):
    """
    Logs records to Argilla. 
    Expects 'gliner_predictions' to be a list of dicts with 'start', 'end', 'label'.
    """
    def extract_text(row):
        fields = row.get("fields", {})
        return fields.get("context_sheet_description_field", "") if isinstance(fields, dict) else ""
    
    # Extract Context Sheet ID for Metadata
    def extract_csd_id(row):
        meta = row.get("metadata", {})
        if isinstance(meta, dict):
            val = meta.get("context_sheet_description_id")
            return int(val) if val is not None else None
        return None

    texts = df.apply(extract_text, axis=1).tolist()
    csd_ids = df.apply(extract_csd_id, axis=1).tolist()
    predictions = df["gliner_predictions"].tolist()

    records = []
    logger.info(f"Building records for {len(df)} items...")
    
    iterator = zip(df.index, texts, csd_ids, predictions)

    for idx, text_val, csd_id, preds_list in tqdm(iterator, total=len(df)):
        try:
            if not text_val or not isinstance(text_val, str): continue

            # Format Suggestions
            formatted_suggestions = []
            if isinstance(preds_list, list):
                for p in preds_list:
                    formatted_suggestions.append({
                        "label": str(p["label"]),
                        "start": int(p["start"]),
                        "end": int(p["end"]),
                        "score": 1.0
                    })

            # Metadata
            record_metadata = {}
            if csd_id is not None:
                record_metadata["context_sheet_description_id"] = csd_id

            record = rg.Record(
                fields={"sentence_field": str(text_val)},
                metadata=record_metadata,
                suggestions=[
                    rg.Suggestion(
                        question_name="entities",
                        value=formatted_suggestions,
                        agent="GLiNER-2.0-Large"
                    )
                ]
            )
            records.append(record)
        except Exception:
            traceback.print_exc()

    if records:
        try:
            dataset.records.log(records)
            logger.info("Records logged successfully.")
        except Exception:
            logger.error("Failed to log records.", exc_info=True)

In [ ]:
@dataclass
class PipelineConfig:
    recreate_workspaces: bool
    recreate_schema: bool
    reprocess_text: bool
    upload_records: bool
    cache_path: Path
    test_subset_size: Optional[int] = None  # New parameter for testing

In [ ]:
def run_pipeline(cfg: PipelineConfig, env_vars: dict):
    logger.info("Starting pipeline execution...")
    
    # --- Step 1: Environment & Client Setup ---
    try:
        client = configure_argilla_client(env_vars=env_vars)
    except Exception:
        logger.error("Failed to initialize Argilla client.", exc_info=True)
        return

    if cfg.recreate_workspaces:
        logger.info("Recreating workspaces and users...")
        configure_argilla_resources(client, env_vars)

    # --- Step 2: Dataset Schema Management ---
    workspace_name = ast.literal_eval(env_vars["ARGILLA_WORKSPACES"])[0]["name"]
    dataset_name = env_vars["ARGILLA_DATASET"]

    # Load guidelines
    guidelines_path = DATA_DIR / "archaeobert_ner_gudelines_mt_translation.md"
    guidelines = ""
    if guidelines_path.exists():
        with open(guidelines_path) as inf:
            guidelines = inf.read()

    dataset = None
    if cfg.recreate_schema:
        dataset = setup_argilla_dataset(client, dataset_name, workspace_name, guidelines)
    else:
        try:
            dataset = client.datasets(name=dataset_name, workspace=workspace_name)
            logger.info(f"Using existing dataset: {dataset_name}")
        except Exception:
            logger.warning(f"Dataset {dataset_name} not found. Consider setting recreate_schema=True.")

    # --- Step 3: Data Processing (With Caching) ---
    df_annotation = None
    
    if cfg.reprocess_text:
        logger.info("Status: RUNNING NEW INFERENCE")
        guidelines_dict = get_annotation_guidelines()

        # 1. Load Data
        df = get_dataset_as_dataframe(
            client=client, 
            dataset_name="atrium_context_sheet_descriptions", 
            workspace_name="atrium"
        )

        if cfg.test_subset_size:
            logger.info(f"Sampling {cfg.test_subset_size} records (80% long, 20% short)...")
            df = sample_diverse_lengths(df, n=cfg.test_subset_size, long_ratio=0.8)
        
        # # Test Subset Logic
        # if cfg.test_subset_size and cfg.test_subset_size > 0:
        #     logger.info(f"TEST MODE: Subsetting dataset to first {cfg.test_subset_size} records.")
        #     df = df.head(cfg.test_subset_size)

        # 2. Run Inference
        df_annotation = annotate_dataframe(
            df=df, 
            extractor=extractor, 
            guidelines_dict=guidelines_dict
        )
        
        # 3. CONVERT & FILTER (Fixes the TypeError and Logic)
        df_annotation = standardize_and_filter_predictions(df_annotation)
        # Apply Corrections (The new step)
        corrections_file = DATA_DIR / "atrium_csd_dataset_corrections.csv"
        df_annotation = patch_predictions_with_corrections(df_annotation, corrections_file)        

        # 3. Save to Cache
        logger.info(f"Saving annotated dataframe to cache: {cfg.cache_path}")
        df_annotation.to_pickle(cfg.cache_path)
        
    else:
        # LOAD FROM CACHE
        logger.info("Status: LOADING FROM CACHE")
        if cfg.cache_path.exists():
            logger.info(f"Loading cached dataframe from: {cfg.cache_path}")
            df_annotation = pd.read_pickle(cfg.cache_path)
            logger.info(f"Loaded {len(df_annotation)} records from cache.")
        else:
            logger.error(f"Cache file not found at {cfg.cache_path}. You must run with reprocess_text=True first.")
            return
    
    
    # --- Step 4: Record Upload ---
    if cfg.upload_records and df_annotation is not None and dataset is not None:
        logger.info("Starting record upload...")
        log_annotation_records(df_annotation, dataset)
    elif cfg.upload_records:
        logger.warning("Skipping upload: DataFrame or Dataset is missing.")

In [ ]:
config = PipelineConfig(
    recreate_workspaces=False, # Keep existing
    recreate_schema=True,      # Reset schema to be safe
    reprocess_text=True,       # TRUE = Run GLiNER
    upload_records=True,
    cache_path=DATA_DIR / "atrium_csd_ner_processed_dataframe_cache.pkl",
    test_subset_size=100
)
run_pipeline(config, env_vars)

In [ ]:
config = PipelineConfig(
    recreate_workspaces=False,
    recreate_schema=True,      # Rebuild empty dataset
    reprocess_text=False,      # FALSE = Load from Cache
    upload_records=True,       # Try uploading again
    cache_path=DATA_DIR / "atrium_csd_ner_processed_dataframe_cache.pkl",
    test_subset_size=100
)
run_pipeline(config, env_vars)


In [ ]:

def _check_overlap(current_entity: dict, all_entities: list[dict]) -> bool:
    """
    Checks if the current entity spatially overlaps with any other entity 
    in the list.
    
    Overlap condition: start1 < end2 AND start2 < end1
    """
    s1, e1 = current_entity['start'], current_entity['end']
    
    for other in all_entities:
        # Skip comparing with self (using identity or exact coordinate match)
        if other is current_entity:
            continue
            
        s2, e2 = other['start'], other['end']
        
        # Check for any overlap type (partial, nested, enclosing)
        if s1 < e2 and s2 < e1:
            return True
            
    return False

def _get_token_context(text: str, start: int, end: int, window: int = 5) -> str:
    """
    Extracts N space-separated tokens before and after the entity.
    """
    # Extract text segments
    prefix = text[:start]
    suffix = text[end:]
    
    # Split into tokens (whitespace splitting)
    prefix_tokens = prefix.strip().split()
    suffix_tokens = suffix.strip().split()
    
    # Slice window
    left_context = " ".join(prefix_tokens[-window:])
    right_context = " ".join(suffix_tokens[:window])
    
    return f"{left_context} [TARGET] {right_context}"

def flatten_entity_results(df: pd.DataFrame, window_size: int = 5) -> pd.DataFrame:
    """
    Explodes the document-level predictions into an entity-level DataFrame.
    
    Args:
        df: DataFrame containing 'fields' (dict) and 'gliner_predictions' (list of dicts).
        window_size: Number of tokens to include on left/right of entity.
        
    Returns:
        pd.DataFrame: Flattened dataframe with one row per entity.
    """
    flattened_records = []
    
    logger.info(f"Flattening entities from {len(df)} documents...")
    
    for idx, row in df.iterrows():
        # 1. Retrieve Text
        fields_data = row.get("fields", {})
        text = None
        if isinstance(fields_data, dict):
            text = fields_data.get("context_sheet_description_field")
            
        # 2. Retrieve Predictions
        predictions = row.get("gliner_predictions", [])
        
        # Skip if data is missing or empty
        if not text or not isinstance(text, str) or not predictions:
            continue
            
        original_id = row.get("id", idx)
        
        # 3. Process each entity
        for entity in predictions:
            try:
                # Determine Overlap
                has_overlap = _check_overlap(entity, predictions)
                
                # Extract Context
                context_str = _get_token_context(
                    text, 
                    entity['start'], 
                    entity['end'], 
                    window=window_size
                )
                
                # Build Record
                record = {
                    "original_doc_id": original_id,
                    "entity_text": entity["text"],
                    "entity_label": entity["label"],
                    "score": entity.get("score", 0.0),
                    "start": entity["start"],
                    "end": entity["end"],
                    "context": context_str,
                    "has_overlap": has_overlap
                }
                flattened_records.append(record)
                
            except Exception as e:
                logger.error(f"Error processing entity in doc {original_id}: {e}")
                continue

    # Create DataFrame
    entity_df = pd.DataFrame(flattened_records)
    
    logger.info(f"Generated flattened DataFrame with {len(entity_df)} entities.")
    return entity_df


def load_and_flatten_cache(cache_path: Path) -> pd.DataFrame:
    """
    Loads the cached DataFrame and flattens entities for analysis.
    """
    if not cache_path.exists():
        logger.error(f"Cache file not found at: {cache_path}")
        return pd.DataFrame()

    try:
        logger.info(f"Loading annotations from cache: {cache_path}")
        # Using read_pickle as the config specifies a .pkl file
        df_cached = pd.read_pickle(cache_path)
        
        # Validation: Ensure it is a DataFrame
        if not isinstance(df_cached, pd.DataFrame):
            logger.error("Cached object is not a pandas DataFrame.")
            return pd.DataFrame()

        logger.info(f"Successfully loaded {len(df_cached)} documents.")

        # Apply the flattening function defined previously
        # metrics_df contains: entity_text, label, score, context, has_overlap, etc.
        metrics_df = flatten_entity_results(df_cached, window_size=5)
        
        return metrics_df

    except Exception as e:
        logger.error(f"Failed to process cache: {e}")
        return pd.DataFrame()

# Execution
flat_entities_df = load_and_flatten_cache(config.cache_path)



In [ ]:
flat_entities_df.to_csv("/home/prokopis/Dropbox/atrium_csd_dataset.csv", index=False)  

In [ ]:
import ast

def diagnose_predictions(df):
    logger.info(f"Inspecting DataFrame with {len(df)} rows...")
    
    # 1. Check if column exists
    if "gliner_predictions" not in df.columns:
        logger.error("CRITICAL: 'gliner_predictions' column missing.")
        return

    # 2. Inspect first 3 non-empty rows
    count = 0
    for idx, row in df.iterrows():
        if count >= 3: break
        
        # Extract text exactly as the logger does
        fields = row.get("fields", {})
        text = fields.get("context_sheet_description_field", "") if isinstance(fields, dict) else ""
        preds = row["gliner_predictions"]
        
        print(f"\n--- Record {idx} ---")
        print(f"Text length: {len(text)}")
        print(f"Raw Prediction Type: {type(preds)}")
        print(f"Raw Prediction Content: {str(preds)[:100]}...") # Truncate for readability

        # Simulate the parsing logic
        parsed_preds = []
        try:
            # Case A: Stringified list "[{...}]"
            if isinstance(preds, str):
                print("  -> Detected STRING format. Attempting literal_eval...")
                parsed_preds = ast.literal_eval(preds)
            
            # Case B: List of strings ["{...}"]
            elif isinstance(preds, list) and len(preds) > 0 and isinstance(preds[0], str):
                print("  -> Detected LIST OF STRINGS. Attempting conversion...")
                parsed_preds = [ast.literal_eval(p) if isinstance(p, str) else p for p in preds]
                
            # Case C: Standard List
            elif isinstance(preds, list):
                print("  -> Detected LIST format.")
                parsed_preds = preds
                
        except Exception as e:
            print(f"  -> PARSING ERROR: {e}")

        print(f"Parsed Item Count: {len(parsed_preds)}")
        
        # Validate Offsets
        if parsed_preds:
            first_pred = parsed_preds[0]
            if isinstance(first_pred, dict):
                start = first_pred.get("start")
                end = first_pred.get("end")
                label = first_pred.get("label")
                
                print(f"  -> First Entity: '{label}' ({start}, {end})")
                
                # Check boundaries
                if end > len(text):
                    print(f"  -> CRITICAL WARNING: Entity end ({end}) exceeds text length ({len(text)}). Argilla will DROP this.")
                elif text[start:end] == "":
                    print(f"  -> WARNING: Entity spans empty string.")
                else:
                    print(f"  -> Span text snippet: '{text[start:end]}'")
            else:
                print(f"  -> ERROR: Parsed item is not a dict: {type(first_pred)}")
        else:
            print("  -> NO PREDICTIONS found for this row.")
            
        count += 1

# # Run diagnosis on the dataframe (assuming df_annotation or df exists from previous steps)
# # If you need to reload from cache:
# cache_path = config.cache_path 
# if cache_path.exists():
#     df_debug = pd.read_pickle(cache_path)
#     diagnose_predictions(df_debug)
# else:
#     print("Cache not found. Please run pipeline with reprocess_text=True first.")

In [ ]:

# --- Example Usage ---
# 1. Export the Drafts
drafts_df = export_draft_records(
    dataset_name="atrium_csd_annotations", 
    workspace="atrium", 
    legacy_mode=False # Set True if your source is an old TokenClassification dataset
)

